We know from feature calculator and the linear regression the features which need to be considered (ladder_position, ladder_position_prev and percentage)

First check if algorithm works on ladder_Position_Next

In [17]:
import pandas as pd

df = pd.read_csv("data/train.csv")

df["Year_Num"] = df["Year"].str.replace("x", "0").astype(int)
df["Percentage"] = df["Percentage"].fillna(100)
df["Ladder_Position"] = df["Ladder_Position"].fillna(df["Ladder_Position"].mean())
df["Ladder_Position_Prev"] = df["Ladder_Position_Prev"].fillna(df["Ladder_Position_Prev"].mean())

def predict_ladder_position_next_next(df):
    df = df.copy()

    required_cols = ["Ladder_Position", "Ladder_Position_Prev", "Percentage"]
    if not all(col in df.columns for col in required_cols):
        raise ValueError("Missing one or more required columns.")

    df["Change"] = df["Ladder_Position"] - df["Ladder_Position_Prev"]

    def adjust_change(row):
        percentage = row["Percentage"] / 100
        change = row["Change"]
        factor = percentage if percentage > 1 else (2 - percentage)
        return change * factor

    df["Adjusted_Change"] = df.apply(adjust_change, axis=1)

    df["Ladder_Position_Next_Next_Pred"] = df["Ladder_Position"] + df["Adjusted_Change"]

    df["Ladder_Position_Next_Next_Pred"] = df["Ladder_Position_Next_Next_Pred"].round().clip(lower=1)

    return df[[
        "ID", "Year", "Ladder_Position", "Ladder_Position_Prev",
        "Percentage", "Ladder_Position_Next_Next_Pred"
    ]]

predicted_df = predict_ladder_position_next_next(df)

print(predicted_df[['Ladder_Position_Next_Next_Pred']])
print("Ladder position next: \n", df['Ladder_Position_Next'])
print("Avg off:", df['Ladder_Position_Next'].sum()/predicted_df[['Ladder_Position_Next_Next_Pred']].sum())


      Ladder_Position_Next_Next_Pred
0                                1.0
1                                1.0
2                                1.0
3                               12.0
4                                1.0
...                              ...
1098                             8.0
1099                             8.0
1100                            14.0
1101                            12.0
1102                             1.0

[1103 rows x 1 columns]
Ladder position next: 
 0        1
1       12
2        4
3       10
4        2
        ..
1098     7
1099     7
1100     3
1101    17
1102     7
Name: Ladder_Position_Next, Length: 1103, dtype: int64
Avg off: Ladder_Position_Next_Next_Pred    0.920825
dtype: float64


Lets predict for the season after

In [18]:
import pandas as pd

df = pd.read_csv("data/train.csv")

df["Year_Num"] = df["Year"].str.replace("x", "0").astype(int)
df["Percentage"] = df["Percentage"].fillna(100)
df["Ladder_Position"] = df["Ladder_Position"].fillna(df["Ladder_Position"].mean())
df["Ladder_Position_Next"] = df["Ladder_Position_Next"].fillna(df["Ladder_Position_Next"].mean())

def predict_ladder_position_next_next(df):
    df = df.copy()

    required_cols = ["Ladder_Position", "Ladder_Position_Prev", "Percentage"]

    df["Change"] = df["Ladder_Position_Next"] - df["Ladder_Position"]

    def adjust_change(row):
        percentage = row["Percentage"] / 100
        change = row["Change"]
        factor = percentage if percentage > 1 else (2 - percentage)
        return change * factor

    df["Adjusted_Change"] = df.apply(adjust_change, axis=1)

    df["Ladder_Position_Next_Next_Pred"] = df["Ladder_Position_Next"] + df["Adjusted_Change"]

    df["Ladder_Position_Next_Next_Pred"] = df["Ladder_Position_Next_Next_Pred"].round().clip(lower=1)

    return df[[
        "ID", "Year", "Ladder_Position", "Ladder_Position_Next",
        "Percentage", "Ladder_Position_Next_Next_Pred"
    ]]

predicted_df = predict_ladder_position_next_next(df)

print(predicted_df[['Ladder_Position_Next_Next_Pred']])


      Ladder_Position_Next_Next_Pred
0                                1.0
1                               23.0
2                               10.0
3                                7.0
4                                3.0
...                              ...
1098                             9.0
1099                             5.0
1100                             1.0
1101                            27.0
1102                            10.0

[1103 rows x 1 columns]
